# Emotion Detection Pipeline Interface

Run this notebook to test the audio emotion detection system.

In [1]:
import asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output
from orchestrator import EmotionOrchestrator
import os

# Initialize Orchestrator
orchestrator = EmotionOrchestrator()

print("System Initialized.")

In [2]:
# UI Components
current_file_display = widgets.Text(
    value='Ready to start',
    description='Current File:',
    disabled=True,
    layout=widgets.Layout(width='80%')
)

run_btn = widgets.Button(
    description='Start / Next File',
    button_style='primary',
    icon='play'
)

output_area = widgets.Output()

# Feedback Buttons
btn_layout = widgets.Layout(width='150px')
btn_openai = widgets.Button(description='OpenAI Better', button_style='success', layout=btn_layout)
btn_local = widgets.Button(description='Local Better', button_style='info', layout=btn_layout)
btn_same = widgets.Button(description='Same / Equal', button_style='warning', layout=btn_layout)
btn_bad = widgets.Button(description='Both Bad', button_style='danger', layout=btn_layout)

feedback_box = widgets.HBox([btn_openai, btn_local, btn_same, btn_bad])
feedback_box.layout.visibility = 'hidden'

current_result = None

def show_results(result):
    global current_result
    current_result = result
    
    # Update display with filename
    if "file" in result:
        current_file_display.value = os.path.basename(result['file'])
    
    with output_area:
        clear_output()
        if "error" in result:
            print(f"Error: {result['error']}")
            return

        print(f"File: {os.path.basename(result['file'])}")
        print("-" * 40)
        print("OPENAI ANALYSIS:")
        print(f"Emotion: {result['openai'].get('emotion', 'N/A')}")
        print(f"Confidence: {result['openai'].get('confidence', 'N/A')}")
        print("-" * 40)
        print("LOCAL MODEL ANALYSIS:")
        # Local result is a list of dicts, take top 1
        if result['local'] and len(result['local']) > 0:
            top = result['local'][0]
            print(f"Emotion: {top['label']}")
            print(f"Score: {top['score']:.4f}")
        else:
            print("No result.")
        print("-" * 40)
        print("Please vote on the quality:")
    
    feedback_box.layout.visibility = 'visible'

async def on_run_click(b):
    with output_area:
        clear_output()
        print("Looking for next file...")
        feedback_box.layout.visibility = 'hidden'
    
    try:
        # run_analysis(None) triggers auto-selection
        res = await orchestrator.run_analysis()
        show_results(res)
    except Exception as e:
        with output_area:
            print(f"Error: {e}")

def on_feedback_click(pref):
    if current_result:
        orchestrator.save_feedback(current_result, pref)
        with output_area:
            print(f"\nFeedback '{pref}' saved!")
            print("Click 'Start / Next File' to continue.")
        feedback_box.layout.visibility = 'hidden'

run_btn.on_click(lambda b: asyncio.create_task(on_run_click(b)))

btn_openai.on_click(lambda b: on_feedback_click("openai"))
btn_local.on_click(lambda b: on_feedback_click("local"))
btn_same.on_click(lambda b: on_feedback_click("same"))
btn_bad.on_click(lambda b: on_feedback_click("bad"))

display(widgets.VBox([current_file_display, run_btn, output_area, feedback_box]))